# Spam Classification: CountVectorizer & TF-IDF with Naive Bayes and SVM

This notebook:
- Loads the cleaned spam dataset
- Uses a **single train/test split** for every comparison
- Fits **CountVectorizer** and **TF-IDF** vectorizers on the training data only, then transforms the test data
- Trains **Multinomial Naive Bayes** and **Linear SVM (LinearSVC)** on both feature sets
- Checks each model for **overfitting / underfitting**
- Reports **accuracy, precision, recall, F1** for every combination


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
import joblib

RANDOM_STATE = 42

## 1. Load the dataset

In [2]:
df = pd.read_csv("../data/cleaned_spam.csv")
df = df.dropna(subset=["clean_message", "label"]).reset_index(drop=True)

X = df["clean_message"].astype(str)
y = df["label"].map({"ham": 0, "spam": 1})

print("Dataset shape:", df.shape)
print("Class distribution:")
print(df["label"].value_counts())
df.head()

Dataset shape: (5167, 3)
Class distribution:
label
ham     4514
spam     653
Name: count, dtype: int64


,label,message,clean_message
0,ham,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,ham,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah i don t think he goes to usf he lives arou...


## 2. Train/test split

**This exact split is reused for every model and every feature configuration below**, so all comparisons are apples-to-apples.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y )
print("Train size:", len(X_train), "| Test size:", len(X_test))

Train size: 4133 | Test size: 1034


## 3. Feature extraction: CountVectorizer & TF-IDF

Both vectorizers are **fit on the training data only** and then used to **transform** the test data (no data leakage). Both use the same configuration (stop words removed, unigrams+bigrams, `min_df=2`, `max_df=0.95`) so the comparison isolates the effect of the vectorizer, not the hyperparameters.

In [4]:
count_vec = CountVectorizer(stop_words="english",ngram_range=(1, 2),min_df=2,max_df=0.95)

X_train_count = count_vec.fit_transform(X_train)
X_test_count = count_vec.transform(X_test)


tfidf_vec = TfidfVectorizer(stop_words="english",ngram_range=(1, 2),min_df=2,max_df=0.95)

X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf = tfidf_vec.transform(X_test)

print("CountVectorizer vocab size:", len(count_vec.vocabulary_))
print("TF-IDF vocab size:", len(tfidf_vec.vocabulary_))


feature_sets = {
    "CountVectorizer": (X_train_count, X_test_count),
    "TF-IDF": (X_train_tfidf, X_test_tfidf)
}

CountVectorizer vocab size: 5583
TF-IDF vocab size: 5583


## 4. Models

Same `random_state`, same split, same feature config for every run.

In [5]:
def get_models():
    return {
        "MultinomialNB": MultinomialNB(alpha=0.5),
        "LinearSVC": LinearSVC(C=1.0,random_state=RANDOM_STATE, class_weight="balanced", max_iter=5000),
    }


In [6]:
models = get_models()

trained_models = {}

## 5. Overfitting / underfitting check function

Compares train accuracy, test accuracy, and 5-fold cross-validation accuracy:

- **Underfitting**: train accuracy itself is low (< 0.85)
- **Overfitting**: train accuracy is high but the train-test gap is large (> 0.05)
- **Good fit**: train accuracy is high and the train-test gap is small


In [7]:
def check_fit(model, X_train, y_train, X_test, y_test):

    train_accuracy = model.score(X_train, y_train)

    test_accuracy = model.score(X_test, y_test)

    gap = train_accuracy - test_accuracy


    print("Train Accuracy:", round(train_accuracy, 4))
    print("Test Accuracy:", round(test_accuracy, 4))
    print("Train-Test Gap:", round(gap, 4))

    if train_accuracy < 0.85:

        print("Status: Underfitting")

    elif gap > 0.05:

        print("Status: Possible Overfitting")

    else:

        print("Status: Good Fit")

In [8]:
for feature_name, (X_train_features, X_test_features) in feature_sets.items():

    print("=" * 60)
    print("Feature Set:", feature_name)
    print("=" * 60)

    for model_name, model in models.items():

        print("\nModel:", model_name)
        print("-" * 40)


        # Train the model
        model.fit(X_train_features, y_train)


        # Predictions
        y_pred = model.predict(X_test_features)


        # Accuracy
        accuracy = accuracy_score(y_test, y_pred)

        print("Accuracy:", accuracy)


        # Classification Report
        print("\nClassification Report:")

        print(classification_report(y_test,y_pred,target_names=["ham", "spam"]))


        # Save trained model in dictionary
        trained_models[(feature_name, model_name)] = model


        # Run check_fit function
        print("Model Fit Check:")

        check_fit(model,X_train_features,y_train,X_test_features,y_test)

        # Store trained model
        trained_models[(feature_name, model_name)] = model

Feature Set: CountVectorizer

Model: MultinomialNB
----------------------------------------
Accuracy: 0.9835589941972921

Classification Report:
              precision    recall  f1-score   support

         ham       0.99      1.00      0.99       903
        spam       0.97      0.90      0.93       131

    accuracy                           0.98      1034
   macro avg       0.98      0.95      0.96      1034
weighted avg       0.98      0.98      0.98      1034

Model Fit Check:


Train Accuracy: 0.9925
Test Accuracy: 0.9836
Train-Test Gap: 0.0089
Status: Good Fit

Model: LinearSVC
----------------------------------------
Accuracy: 0.9816247582205029

Classification Report:
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       903
        spam       0.97      0.88      0.92       131

    accuracy                           0.98      1034
   macro avg       0.98      0.94      0.96      1034
weighted avg       0.98      0.98      0.98      1034

Model Fit Check:
Train Accuracy: 0.9995
Test Accuracy: 0.9816
Train-Test Gap: 0.0179
Status: Good Fit
Feature Set: TF-IDF

Model: MultinomialNB
----------------------------------------
Accuracy: 0.9825918762088974

Classification Report:
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       903
        spam       1.00      0.86      0.93       131

    accuracy                           0.98      1034
   macro avg       0

Save all trained models

In [9]:
import joblib

for (feature_name, model_name), model in trained_models.items():

    file_name = (
        feature_name.replace("-", "_").lower()
        + "_"
        + model_name.lower()
        + ".pkl"
    )

    joblib.dump(model, "../models/" + file_name)

    print(file_name, "saved successfully")

countvectorizer_multinomialnb.pkl saved successfully
countvectorizer_linearsvc.pkl saved successfully
tf_idf_multinomialnb.pkl saved successfully
tf_idf_linearsvc.pkl saved successfully


Save CountVectorize

In [10]:
joblib.dump(count_vec, "../models/count_vectorizer.pkl")

print("CountVectorizer saved successfully")

CountVectorizer saved successfully


Save TF-IDF Vectorizer

In [11]:
joblib.dump(tfidf_vec, "../models/tfidf_vectorizer.pkl")

print("TF-IDF Vectorizer saved successfully")

TF-IDF Vectorizer saved successfully
